In [ ]:
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec



# Login + Session

In [ ]:
ST_BASE    = "https://www.space-track.org"
ST_LOGIN   = f"{ST_BASE}/ajaxauth/login"
ST_QUERY   = f"{ST_BASE}/basicspacedata/query"

USERNAME = "nathandarby2022@gmail.com"   # replace with your Space-Track credentials
PASSWORD = "Boobear32*Boobear32"             # consider loading from env variable

session = requests.Session()

resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})

if resp.status_code == 200:
    print("Logged in to Space-Track successfully!")
else:
    print(f"Login failed: {resp.status_code}")

In [ ]:
resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
print(resp.status_code)

# SATCAT Fetch + Cache

In [ ]:
import json
import os
from datetime import datetime, timezone

SATCAT_CACHE_FILE = os.path.expanduser("~/satcat_cache.json")

def get_norad_ids(intldes, session, force=False):
    """
    Get all NORAD IDs for a launch group.
    Reads from local cache if available, otherwise queries Space-Track satcat.
    """
    # Check cache first
    if os.path.exists(SATCAT_CACHE_FILE) and not force:
        with open(SATCAT_CACHE_FILE) as f:
            cache = json.load(f)
        
        if cache.get('intldes') == intldes:
            print(f"Loaded {len(cache['norad_ids'])} NORAD IDs from cache")
            print(f"   Launch:    {intldes}")
            print(f"   Cached at: {cache['fetched_at']}")
            for obj in cache['objects']:
                print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
            return cache['norad_ids']
        else:
            print(f"Cache is for different launch ({cache.get('intldes')}) — fetching fresh")

    # Query satcat
    print(f"Querying Space-Track satcat for launch {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    # Save to cache
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'], 
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(SATCAT_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched and cached {len(norad_ids)} payloads for launch {intldes}")
    for obj in objects:
        print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
    
    return norad_ids

# Current TLE Fetch + Cache

In [ ]:
TLE_CACHE_FILE = os.path.expanduser("~/tle_cache.json")

def fetch_tles(norad_ids, session, chunk_size=50, force=False):
    if os.path.exists(TLE_CACHE_FILE) and not force:
        with open(TLE_CACHE_FILE) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            remaining = 3600 - age.total_seconds()
            print(f"Loaded TLEs from Cache")
            print(f"   Last Query:            {int(age.total_seconds()//60)} min ago")
            print(f"   Next Available Query:  {int(remaining//60)} min")
            print(f"   Total TLEs:            {len(cache['tles'])}")
            return cache['tles']
        else:
            print("Cache expired — fetching fresh TLEs")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(TLE_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nLoaded TLEs from Query")
    print(f"   Last Query:            just now")
    print(f"   Next Available Query:  60 min")
    print(f"   Total TLEs:            {len(all_tles)}")
    return all_tles

In [ ]:
# Transporter-9 — Oct 2023
norad_ids = get_norad_ids('2023-174', session, force=False)
tles = fetch_tles(norad_ids, session, force=False)

# Dataframe

In [ ]:
rows = []
for t in tles:
    line1 = t['line1']
    intldes_raw = line1[9:17].strip()
    year  = intldes_raw[:2]
    num   = intldes_raw[2:5]
    piece = intldes_raw[5:]
    full_year = f"19{year}" if int(year) >= 57 else f"20{year}"
    intldes = f"{full_year}-{num}{piece}"
    
    rows.append({
        'International Designator': intldes,
        'NORAD Catalog Number':     line1[2:7].strip(),
        'Name':                     t['name'],
        'TLE Line 1':               line1,
        'TLE Line 2':               t['line2']
    })

df_tles = pd.DataFrame(rows)
print(f"Total objects: {len(df_tles)}")
df_tles

# Historical TLE fetch (gp_history) (!!! Careful !!!)

In [ ]:
import os
os.remove(os.path.expanduser("~/tle_history_cache.json"))
print("Cache cleared")

In [ ]:
# Section 5 — Historical TLE Fetch (gp_history)
# One-time query — never re-query, always load from cache

HISTORY_CACHE_FILE = os.path.expanduser("~/tle_history_cache.json")

with open(SATCAT_CACHE_FILE) as f:
    satcat_cache = json.load(f)

def fetch_historical_tles(norad_ids, start_date, end_date, session, chunk_size=50):
    """
    Fetch historical TLEs from Space-Track gp_history class.
    One-time query only — results cached permanently to disk.
    start_date, end_date: 'YYYY-MM-DD' strings
    """
    # Check cache — never re-query gp_history
    if os.path.exists(HISTORY_CACHE_FILE):
        with open(HISTORY_CACHE_FILE) as f:
            cache = json.load(f)
        print(f"Historical TLEs loaded from cache")
        print(f"   Launch:     {cache['intldes']}")
        print(f"   Epoch range: {cache['start_date']} → {cache['end_date']}")
        print(f"   Records:    {len(cache['tles'])}")
        return cache['tles']

    print(f"Querying gp_history: {start_date} → {end_date}")
    print(f"WARNING: This is a one-time query — results will be cached permanently")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)

        url = (f"{ST_QUERY}/class/gp_history"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/epoch/{start_date}--{end_date}"
               f"/orderby/NORAD_CAT_ID,EPOCH"
               f"/format/json")

        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code} — {len(resp.json()) if resp.status_code == 200 else 0} records")

        if resp.status_code != 200 or not resp.text.strip():
            continue

        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'norad': obj['NORAD_CAT_ID'],
                    'epoch': obj['EPOCH'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache permanently
    cache = {
        'intldes':    satcat_cache['intldes'],
        'start_date': start_date,
        'end_date':   end_date,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'tles':       all_tles
    }
    with open(HISTORY_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nHistorical TLEs fetched and cached permanently")
    print(f"   Total records: {len(all_tles)}")
    print(f"   Cache file:    {HISTORY_CACHE_FILE}")
    return all_tles

# Transporter-9 launched October 6, 2023
# Query first 7 days post-deployment
historical_tles = fetch_historical_tles(
    norad_ids,
    start_date='2023-11-11',
    end_date='2024-01-11',
    session=session
)

# Fetched TLE time to identification analysis (useful for pitches)

In [ ]:
first_catalog_clean = first_catalog.drop_duplicates(subset='norad').sort_values('days_after_launch')
print(f"Unique satellites: {len(first_catalog_clean)}")
print(f"Mean days to catalog: {first_catalog_clean['days_after_launch'].mean():.1f}")
print(f"Range: {first_catalog_clean['days_after_launch'].min():.1f} — {first_catalog_clean['days_after_launch'].max():.1f} days")

In [ ]:


# Check first cataloged date per satellite from historical TLEs
hist_df = pd.DataFrame(historical_tles)
hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)

first_catalog = hist_df.groupby(['norad', 'name'])['epoch'].min().reset_index()
first_catalog.columns = ['norad', 'name', 'first_cataloged']
first_catalog['days_after_launch'] = (first_catalog['first_cataloged'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
first_catalog = first_catalog.sort_values('days_after_launch').reset_index(drop=True)

print(f"{'Name':<25} {'NORAD':<8} {'First Cataloged':<25} {'Days After Launch':>17}")
print("-" * 80)
for _, row in first_catalog.iterrows():
    print(f"{row['name']:<25} {row['norad']:<8} {str(row['first_cataloged']):<25} {row['days_after_launch']:>17.1f}")

print(f"\nSummary:")
print(f"   Earliest cataloged: {first_catalog['days_after_launch'].min():.1f} days — {first_catalog.iloc[0]['name']}")
print(f"   Latest cataloged:   {first_catalog['days_after_launch'].max():.1f} days — {first_catalog.iloc[-1]['name']}")
print(f"   Mean:               {first_catalog['days_after_launch'].mean():.1f} days")

In [ ]:
# Get TBA/OBJECT name and confirmed name separately
tba_names = hist_df[~hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
tba_names.columns = ['norad', 'unconfirmed_name']

confirmed_names = hist_df[hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
confirmed_names.columns = ['norad', 'confirmed_name']

# Merge everything
confirmation_df = first_cataloged.merge(first_confirmed, on='norad', how='left')
confirmation_df = confirmation_df.merge(tba_names, on='norad', how='left')
confirmation_df = confirmation_df.merge(confirmed_names, on='norad', how='left')
confirmation_df['days_to_catalog']  = (confirmation_df['first_cataloged'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['days_to_confirm']  = (confirmation_df['first_confirmed'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['confirmation_lag'] = confirmation_df['days_to_confirm'] - confirmation_df['days_to_catalog']
confirmation_df = confirmation_df.sort_values('days_to_confirm').reset_index(drop=True)

print(f"{'Unconfirmed Name':<25} {'Confirmed Name':<25} {'NORAD':<8} {'Cataloged':>10} {'Confirmed':>10} {'Lag':>8}")
print("-" * 95)
for _, row in confirmation_df.iterrows():
    confirmed     = f"{row['days_to_confirm']:.1f}"  if not pd.isna(row['days_to_confirm'])  else "NOT YET"
    lag           = f"{row['confirmation_lag']:.1f}" if not pd.isna(row['confirmation_lag']) else "N/A"
    confirmed_name = row['confirmed_name'] if not pd.isna(row['confirmed_name']) else "UNCONFIRMED"
    print(f"{row['unconfirmed_name']:<25} {confirmed_name:<25} {row['norad']:<8} {row['days_to_catalog']:>10.1f} {confirmed:>10} {lag:>8}")

print(f"\nSummary:")
print(f"   Mean days to catalog:      {confirmation_df['days_to_catalog'].mean():.1f} days")
print(f"   Mean days to confirm:      {confirmation_df['days_to_confirm'].mean():.1f} days")
print(f"   Mean confirmation lag:     {confirmation_df['confirmation_lag'].mean():.1f} days")
print(f"   Unconfirmed in 30 days:    {confirmation_df['confirmed_name'].isna().sum()}")

In [ ]:
# Check if history cache covers month 2, if not we need to re-query
# First check what we already have
print(f"Current history range: {hist_df['epoch'].min()} → {hist_df['epoch'].max()}")
print(f"Records: {len(hist_df)}")

In [ ]:
# Reload cache
with open(HISTORY_CACHE_FILE) as f:
    history_cache = json.load(f)

historical_tles = history_cache['tles']
hist_df = pd.DataFrame(historical_tles)
hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
hist_df['is_confirmed'] = hist_df['name'].apply(is_confirmed)

# Rerun confirmation analysis
tba_names       = hist_df[~hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
tba_names.columns = ['norad', 'unconfirmed_name']

confirmed_names = hist_df[hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
confirmed_names.columns = ['norad', 'confirmed_name']

first_cataloged = hist_df.groupby('norad')['epoch'].min().reset_index()
first_cataloged.columns = ['norad', 'first_cataloged']

first_confirmed = hist_df[hist_df['is_confirmed']].groupby('norad')['epoch'].min().reset_index()
first_confirmed.columns = ['norad', 'first_confirmed']

confirmation_df = first_cataloged.merge(first_confirmed, on='norad', how='left')
confirmation_df = confirmation_df.merge(tba_names, on='norad', how='left')
confirmation_df = confirmation_df.merge(confirmed_names, on='norad', how='left')
confirmation_df['days_to_catalog']  = (confirmation_df['first_cataloged'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['days_to_confirm']  = (confirmation_df['first_confirmed'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['confirmation_lag'] = confirmation_df['days_to_confirm'] - confirmation_df['days_to_catalog']
confirmation_df = confirmation_df.sort_values('days_to_confirm').reset_index(drop=True)

print(f"{'Unconfirmed Name':<25} {'Confirmed Name':<25} {'NORAD':<8} {'Cataloged':>10} {'Confirmed':>10} {'Lag':>8}")
print("-" * 95)
for _, row in confirmation_df.iterrows():
    confirmed      = f"{row['days_to_confirm']:.1f}"  if not pd.isna(row['days_to_confirm'])  else "NOT YET"
    lag            = f"{row['confirmation_lag']:.1f}" if not pd.isna(row['confirmation_lag']) else "N/A"
    confirmed_name = row['confirmed_name'] if not pd.isna(row['confirmed_name']) else "UNCONFIRMED"
    print(f"{row['unconfirmed_name']:<25} {confirmed_name:<25} {row['norad']:<8} {row['days_to_catalog']:>10.1f} {confirmed:>10} {lag:>8}")

print(f"\nSummary:")
print(f"   Total unique objects:      {len(confirmation_df)}")
print(f"   Mean days to catalog:      {confirmation_df['days_to_catalog'].mean():.1f} days")
print(f"   Mean days to confirm:      {confirmation_df['days_to_confirm'].mean():.1f} days")
print(f"   Mean confirmation lag:     {confirmation_df['confirmation_lag'].mean():.1f} days")
print(f"   Confirmed in 60 days:      {confirmation_df['confirmed_name'].notna().sum()}")
print(f"   Unconfirmed in 60 days:    {confirmation_df['confirmed_name'].isna().sum()}")

In [ ]:
unconfirmed = confirmation_df[confirmation_df['confirmed_name'].isna()]
print(f"Still unconfirmed after 60 days:")
for _, row in unconfirmed.iterrows():
    print(f"   {row['norad']} — {row['unconfirmed_name']}")

In [ ]:
unconfirmed_norads = ['58258', '58266', '58269', '58287', '58326', '58330', 
                       '58339', '58343', '58344', '58345', '58563', '58564', 
                       '58566', '58572', '58690', '58697']

# Check against current TLE dataframe
print("Unconfirmed objects — current names:")
print(f"{'NORAD':<8} {'Current Name':<30}")
print("-" * 40)
for norad in unconfirmed_norads:
    match = df_tles[df_tles['NORAD Catalog Number'] == norad]
    if len(match) > 0:
        print(f"{norad:<8} {match.iloc[0]['Name']:<30}")
    else:
        print(f"{norad:<8} NOT IN CURRENT CATALOG")

In [ ]:
# Query satcat for decay dates of unconfirmed objects
unconfirmed_gone = ['58266', '58269', '58330', '58339', '58343', 
                     '58344', '58345', '58563', '58564', '58566', 
                     '58572', '58690', '58697']

ids_str = ','.join(unconfirmed_gone)
url = f"{ST_QUERY}/class/satcat/NORAD_CAT_ID/{ids_str}/format/json"
resp = session.get(url)
for obj in resp.json():
    print(f"{obj['NORAD_CAT_ID']:<8} {obj['SATNAME']:<30} Decay: {obj['DECAY']}")